# Lab 03-01 — FAISS: the in-memory vector index

**Track 03 · Vector databases** — a vector database stores embeddings and answers one question: "which stored vectors are closest to this query vector?" FAISS (Facebook AI Similarity Search) is the workhorse of the in-memory end of that spectrum: the index lives entirely in RAM, builds in seconds, and searches exactly.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the scored queries, and the MMR re-ranking all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The pipeline, drawn inline:

```
passages.parquet -> HuggingFaceEmbeddings (BGE, local) -> FAISS IndexFlatL2
   -> similarity_search_with_score_by_vector -> top-3 per question
   -> max_marginal_relevance_search_by_vector -> top-5 diverse (MMR)
```

This lab inspects the three things that decide how a vector database behaves:

* **INDEX TYPE** — FAISS ships dozens; the default here is `IndexFlatL2`, a brute-force exact index: the query vector is compared against every stored vector. Exact search is the gold standard that every approximate method (IVF, HNSW, ...) is measured against. No training, no parameters, just a matrix of vectors and a loop.
* **SCORE CONVENTION** — `IndexFlatL2` reports the SQUARED Euclidean distance between the query and each passage vector: LOWER is more similar (a perfect match scores 0.0). Cosine-based stores (Qdrant — lab 03) flip both the scale and the direction. For unit-norm vectors the two are linked by `cos = 1 - sqL2/2`, which is exactly the number lab 03 will reproduce.
* **PERSISTENCE MODEL** — FAISS is in-memory: build the index, query it, and when the process exits the index is gone (there is no on-disk format until you write one). That tradeoff is the takeaway — compare with Chroma's persistent store in lab 02.

The lab also runs MMR (Maximum Marginal Relevance) once: FAISS's `max_marginal_relevance_search` re-ranks candidates to trade pure similarity for diversity, so a query can surface passages about different aspects of the topic instead of near-duplicates.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `sentence-transformers`, `faiss-cpu`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# Several scratch notebooks may run in parallel on this machine; keep the
# BLAS thread pool small so embedding does not thrash memory.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes a deterministic head of the 3200-passage rag-mini-wikipedia corpus (no randomness, reproducible runs); `QUESTION_IDS = [1606, 1610, 1604]` are real questions from `test.parquet` whose answers live inside the subset; `TOP_K = 3` is the per-question hit list, `MMR_K = 5` the MMR list, and `LAMBDA_MULT = 0.5` the MMR relevance/diversity balance (1.0 = pure similarity, 0.0 = pure diversity). `PREVIEW` truncates the passage previews the demo prints next to each hit, and `FLOAT_BYTES = 4` turns the index dimensions into an in-memory size estimate.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
TOP_K = 3
MMR_K = 5
LAMBDA_MULT = 0.5  # MMR: 1.0 = pure similarity, 0.0 = pure diversity
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
FLOAT_BYTES = 4  # float32


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` rows of `passages.parquet` and returns `(passage_texts, passage_ids)` — the ids are the parquet row indices, which is how the demo and the gate refer back to a hit. `load_questions` pulls the requested `test.parquet` rows as `(question_id, question_text)` pairs. Two small helpers round out the section: `preview` flattens a passage onto one line for printing, and `passage_lookup` maps a passage id back to its text so a hit id can be turned into printable text.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def passage_lookup(texts: list[str], ids: list[int]) -> dict[int, str]:
    """Map passage id -> text, for turning a hit id back into printable text."""
    return dict(zip(ids, texts))


## 3. Experiment — embed, index, query; returns every artifact the demo and the verification gate need

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 100 passages once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. Queries are `similarity_search_with_score_by_vector` at `TOP_K = 3` (the flat-L2 index returns squared-L2 distances, lower = more similar), and MMR is the store's native `max_marginal_relevance_search_by_vector` at `MMR_K = 5` with `lambda_mult = 0.5`. This is the same mechanism the shared `src/vectordb/faiss.py` class wraps.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, index, query; returns every artifact the demo and
#    the verification gate need (no re-computation between the two paths)
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed the whole subset once (batched) + each question once ---------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    question_texts = [qtext for _, qtext in questions]
    query_vecs = [embedder.embed_query(q) for q in question_texts]

    # --- Build the FAISS index (in-memory) ----------------------------------
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs)
    )
    index_s = time.perf_counter() - t0

    # --- Query each question with scores ------------------------------------
    scored = [
        store.similarity_search_with_score_by_vector(qvec, k=TOP_K)
        for qvec in query_vecs
    ]

    # --- MMR on the first question ------------------------------------------
    mmr_docs = store.max_marginal_relevance_search_by_vector(
        query_vecs[0], k=MMR_K, lambda_mult=LAMBDA_MULT
    )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "query_vecs": query_vecs,
        "embed_s": embed_s,
        "index_s": index_s,
        "scored": scored,
        "mmr_docs": mmr_docs,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the corpus subset with the questions; the embed + index timings with the in-memory size estimate; the top-3 per question with squared-L2 scores (lower = more similar); the MMR re-ranking of the first question; then a takeaway explaining why FAISS's flat-L2 index is the honest baseline for lab 04's benchmark — and why everything vanishes at process exit.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    passage_lk = passage_lookup(exp["passage_texts"], exp["passage_ids"])
    n_bytes = exp["indexed"] * exp["dim"] * FLOAT_BYTES

    print("=" * 66)
    print("Lab 01 — FAISS: the in-memory vector index")
    print(f"{BGE_MODEL_NAME} | exact flat-L2 index | in-memory only")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Embed + index:")
    print(f"    embedded {exp['indexed']} passages in {exp['embed_s']:.2f}s (dim {exp['dim']})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s")
    print(f"    in-memory size ~ {n_bytes / 1024:.0f} KB "
          f"({exp['indexed']} x {exp['dim']} x {FLOAT_BYTES}B float32)")

    print(f"\n[3] Top-{TOP_K} per question (score = squared L2 distance, LOWER = more similar):")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for doc, score in exp["scored"][i]:
            pid = doc.metadata.get("id", "?")
            print(f"      {score:8.4f}  [passage {pid}] {preview(doc.page_content)}")
        if i == 0:
            print("      ^ note: 0.0 would be a perfect match; these distances grow")
            print("        as relevance drops")

    print(f"\n[4] MMR on Q[{exp['questions'][0][0]}] (lambda_mult={LAMBDA_MULT}, k={MMR_K}):")
    for rank, doc in enumerate(exp["mmr_docs"], 1):
        pid = doc.metadata.get("id", "?")
        print(f"      {rank}. [passage {pid}] {preview(doc.page_content)}")
    print("      MMR re-ranks the candidates: raise lambda_mult toward 1.0 for")
    print("      pure relevance, lower it toward 0.0 for pure diversity.")

    print("\n[5] Takeaway")
    print("    FAISS's default flat-L2 index is exact, in-RAM, and reports")
    print("    squared-L2 distances (lower = better). It is the fastest store")
    print("    to build and the honest baseline for lab 04's benchmark — but")
    print("    everything vanishes at process exit. Chroma (lab 02) trades a")
    print("    few milliseconds for an on-disk store; Qdrant (lab 03) trades")
    print("    them for a cosine score and a full query language.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: embedding dimension 768 (BGE base), exactly `N_PASSAGES` passages indexed, every question returns exactly `TOP_K` scored hits, squared-L2 scores ascend per query, the two content checks (Q1610's top-1 names the Spanish founder of Montevideo; Q1606's top-1 mentions Montevideo), and MMR returns `MMR_K` distinct documents. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Dimension and count match the model / subset.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Every question returned exactly TOP_K scored hits.
    checks.append(
        ("each question returns TOP_K scored hits",
         all(len(hits) == TOP_K for hits in exp["scored"]))
    )

    # Squared-L2 scores ascend with rank (0.0 would be a perfect match).
    scores_ascending = all(
        [s for _, s in hits] == sorted(s for _, s in hits) for hits in exp["scored"]
    )
    checks.append(("squared-L2 scores ascend per query (lower = more similar)", scores_ascending))

    # Content check: Q1610 "Who founded Montevideo?" must rank the passage
    # that says the Spanish founded Montevideo at #1 (passage id 2 lives
    # inside the first N_PASSAGES).
    q1610_top = exp["scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))

    # Q1606 "Is Uruguay's capital Montevideo?" must rank an Uruguay passage
    # that mentions Montevideo at #1.
    q1606_top = exp["scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    # MMR returns exactly MMR_K distinct documents (no duplicates).
    mmr_texts = [d.page_content for d in exp["mmr_docs"]]
    checks.append(("MMR returns MMR_K distinct documents", len(mmr_texts) == MMR_K and len(set(mmr_texts)) == MMR_K))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes of embedding + index build on rag-mini-wikipedia — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The corpus subset with embedding/index timings, the top-3 per question with squared-L2 scores, and the MMR re-ranking of the first question.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
